# Image Classification using CNN

*   Creating deep networks using Keras
*   Training a neural network
*   Prediction and performance analysis using neural networks
*   Using pre-trained networks
*   Feature visualizations
---

# **Colaboratory environment**
By default, Colab notebooks run on CPU.
You can switch your notebook to run with GPU.

In order to obtain access to the GPU, you need to choose the tab Runtime and then select “Change runtime type” as shown in the following figure:

![Changing runtime](https://miro.medium.com/max/747/1*euE7nGZ0uJQcgvkpgvkoQg.png)

When a pop-up window appears select GPU. Ensure “Hardware accelerator” is set to GPU.

In [ ]:
# Loading necessary libraries (Feel free to add new libraries if you need for any computation)
import numpy as np
from matplotlib import pyplot as plt
from tensorflow import keras

# **Working with a new dataset: CIFAR-10**

The CIFAR-10 dataset consists of 60000 32x32 colour images in 10 classes, with 6000 images per class. There are 50000 training images and 10000 test images. More information about CIFAR-10 can be found [here](https://www.cs.toronto.edu/~kriz/cifar.html).

In Keras, the CIFAR-10 dataset is also preloaded in the form of four Numpy arrays. x_train and y_train contain the training set, while x_test and y_test contain the test data. The images are encoded as Numpy arrays and their corresponding labels ranging from 0 to 9.

Step:

*   Visualize the images in CIFAR-10 dataset. Create a 10 x 10 plot showing 10 random samples from each class.
*   Convert the labels to one-hot encoded form.
*   Normalize the images.




In [ ]:
# CIFAR-10 from the Hugging Face Hub (much faster than keras.datasets.cifar10.load_data())
# The datasets library is preinstalled in Colab; elsewhere: pip install datasets
from datasets import load_dataset

cifar = load_dataset("uoft-cs/cifar10").with_format("numpy")
# same arrays as keras.datasets.cifar10: uint8 images (n, 32, 32, 3) and labels (n, 1), same class order
# [:] turns the whole split into a dict of NumPy arrays (ds["img"] is a lazy Column in datasets 4.x)
train, test = cifar["train"][:], cifar["test"][:]
x_train, y_train = train["img"], train["label"].reshape(-1, 1)
x_test, y_test = test["img"], test["label"].reshape(-1, 1)
print(cifar["train"].features["label"].names)

print(x_train.shape)
print(x_train.ndim)
print(x_train.dtype)

print(x_test.shape)
print(np.unique(y_train, return_counts=True))

In [ ]:
fig, axes = plt.subplots(ncols=10, nrows=10, figsize=(15, 15))
for i in range(10):
  for j in range(10):
    axes[i][j].imshow(x_train[(i * 10) + j], cmap=plt.get_cmap('gray'))
    axes[i][j].axis('off')
plt.show()

x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

print(x_train.shape)
print(x_test.shape)

from tensorflow.keras.utils import to_categorical

y_train = to_categorical(y_train, num_classes=10)
y_test = to_categorical(y_test, num_classes=10)

print(y_train.shape)
print(y_test.shape)

## Define the following model

**For the convolutional front-end, start with a single convolutional layer with a small filter size (3,3) and a modest number of filters (32) followed by a max pooling layer. Use the input as (32,32,3). The filter maps can then be flattened to provide features to the classifier. Use a dense layer with 100 units before the classification layer (which is also a dense layer with softmax activation).**

In [ ]:
from keras.backend import clear_session
clear_session()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input

model = Sequential()
model.add(Input(shape=(32, 32, 3)))
model.add(Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform'))
model.add(MaxPooling2D((2, 2)))
model.add(Flatten())
model.add(Dense(100, activation='relu', kernel_initializer='he_uniform'))
model.add(Dense(10, activation='softmax'))

model.summary()

*   Compile the model using categorical_crossentropy loss, SGD optimizer and use 'accuracy' as the metric.
*   Use the above defined model to train CIFAR-10 and train the model for 512 epochs with a batch size of 32.

In [ ]:
# solution
model.compile(loss='categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])
history_shallow = model.fit(x_train, y_train, epochs=50, batch_size=32, validation_split=0.1)


In [ ]:
plt.subplot(211)
plt.title('Cross Entropy Loss')
plt.plot(history_shallow.history['loss'], color='blue', label='train')

plt.subplot(212)
plt.title('Classification Accuracy')
plt.plot(history_shallow.history['accuracy'], color='green', label='train')
plt.show()

In [ ]:
# Set 8.1: how the models are saved (run this even if you upload models trained earlier into set8/)
import json
import os

OUTPUT = "set8"
os.makedirs(OUTPUT, exist_ok=True)
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

def save_model(name, model, history, n_images, input_size, preprocessing):
    model.save_weights(f"{OUTPUT}/{name}.weights.h5")
    model.save(f"{OUTPUT}/{name}.keras")
    n_train = int(n_images * 0.9)  # fit() used validation_split=0.1
    passport = {
        "model": name,
        "dataset": "CIFAR-10",
        "sample_size": {"train": n_train, "validation": n_images - n_train},
        "class_names": CLASS_NAMES,
        "input_size": input_size,
        "preprocessing": preprocessing,
        "training_history": history.history,
    }
    with open(f"{OUTPUT}/{name}.passport.json", "w") as f:
        json.dump(passport, f, indent=2, default=float)

In [ ]:
# Set 8.1: save the small CNN
save_model("small_cnn", model, history_shallow, len(x_train), [32, 32],
           "RGB image resized to 32x32, pixel values divided by 255.")

## Defining Deeper Architectures: VGG Models

*   Define a deeper model architecture for CIFAR-10 dataset and train the new model for 512 epochs with a batch size of 32. We will use VGG model as the architecture.

**Stack two convolutional layers with 32 filters, each of 3 x 3. Use a max pooling layer and next flatten the output of the previous layer and add a dense layer with 128 units before the classification layer. For all the layers, use ReLU activation function. Use same padding for the layers to ensure that the height and width of each layer output matches the input**


In [ ]:
from keras.backend import clear_session
clear_session()

In [ ]:
# solution
deeper_model = Sequential()
deeper_model.add(Input(shape=(32, 32, 3)))
deeper_model.add(Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
deeper_model.add(Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
deeper_model.add(MaxPooling2D((2, 2)))
deeper_model.add(Flatten())
deeper_model.add(Dense(128, activation='relu', kernel_initializer='he_uniform'))
deeper_model.add(Dense(10, activation='softmax'))

deeper_model.summary()


*   Compile the model using categorical_crossentropy loss, SGD optimizer and use 'accuracy' as the metric.
*   Use the above defined model to train CIFAR-10 and train the model for 512 epochs with a batch size of 32.

In [ ]:
# solution
deeper_model.compile(loss='categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])
history_deeper = deeper_model.fit(x_train, y_train, epochs=50, batch_size=32, validation_split=0.1)

*   Compare the performance of both the models by plotting the loss and accuracy curves of both the training steps. Does the deeper model perform better?


In [ ]:
# solution

plt.subplot(211)
plt.title('Cross Entropy Loss')
plt.plot(history_shallow.history['loss'], color='blue', label='tutorial')
plt.plot(history_deeper.history['loss'], color='red', label='deep')

plt.subplot(212)
plt.title('Classification Accuracy')
plt.plot(history_shallow.history['accuracy'], color='green', label='tutorial')
plt.plot(history_deeper.history['accuracy'], color='red', label='deep')
plt.show()

*   Use predict function to predict the output for the test split
*   Plot the confusion matrix for the new model and comment on the class confusions.


In [ ]:
# solution
predictions = deeper_model.predict(x_test)

print(predictions.shape)
predictions = np.argmax(predictions, axis=1)
print(predictions.shape)
y_test = np.argmax(y_test, axis=1)
print(y_test.shape, predictions.shape)


In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, predictions)
plt.imshow(cm, cmap=plt.cm.Blues)
# put numbers on the cells
thresh = cm.max() / 2.
for i, j in np.ndindex(cm.shape):
    plt.text(j, i, cm[i, j],
             horizontalalignment="center",
             color="white" if cm[i, j] > thresh else "black")
plt.title('Confusion matrix')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.colorbar()
plt.show()

*    Print the test accuracy for the trained model.

In [ ]:
# solution
y_test = to_categorical(y_test, num_classes=10)
test_loss, test_acc = model.evaluate(x_test, y_test)
print('Test accuracy:', test_acc)
print('Test loss:', test_loss)


## Define the complete VGG architecture.

**Stack two convolutional layers with 64 filters, each of 3 x 3 followed by max pooling layer. Stack two more convolutional layers with 128 filters, each of 3 x 3, followed by max pooling, follwed by two more convolutional layers with 256 filters, each of 3 x 3, followed by max pooling. Flatten the output of the previous layer and add a dense layer with 128 units before the classification layer. For all the layers, use ReLU activation function. Use same padding for the layers to ensure that the height and width of each layer output matches the input**

*   Change the size of input to 64 x 64.

In [ ]:
from keras.backend import clear_session
clear_session()

In [ ]:
# solution



*   Compile the model using categorical_crossentropy loss, SGD optimizer and use 'accuracy' as the metric.
*   Use the above defined model to train CIFAR-10 and train the model for 100 epochs with a batch size of 32.
*   Predict the output for the test split and plot the confusion matrix for the new model and comment on the class confusions.

In [ ]:
# solution



## Set 8: a stronger CNN for the feature projections

The small CNN above reaches only about 60 % test accuracy, so its features do not separate the classes and the
t-SNE / UMAP projections look like one cloud. This compact residual network (ResNet-style blocks with batch
normalization), trained with data augmentation, AdamW, a cosine learning rate and label smoothing, reaches roughly
90 % and gives well-separated class clusters. It takes about 15 minutes on a T4 GPU.

In [ ]:
from keras.backend import clear_session
clear_session()

In [ ]:
from tensorflow.keras import layers

def residual_block(x, filters, stride):
    shortcut = x
    x = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    if stride != 1 or shortcut.shape[-1] != filters:  # match the shape of the shortcut
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    return layers.ReLU()(layers.Add()([x, shortcut]))

inputs = keras.Input(shape=(32, 32, 3))
x = layers.RandomFlip("horizontal")(inputs)  # augmentation, only active while training
x = layers.RandomTranslation(0.1, 0.1, fill_mode="reflect")(x)
x = layers.Conv2D(32, 3, padding="same", use_bias=False, kernel_initializer="he_normal")(x)
x = layers.BatchNormalization()(x)
x = layers.ReLU()(x)
for filters, stride in [(32, 1), (64, 2), (64, 1), (128, 2), (128, 1), (256, 2)]:
    x = residual_block(x, filters, stride)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation="relu", kernel_initializer="he_normal", name="features")(x)  # extracted in Set 8.3
outputs = layers.Dense(10, activation="softmax")(x)
strong_cnn = keras.Model(inputs, outputs, name="strong_cnn")
strong_cnn.summary()

In [ ]:
EPOCHS = 40
BATCH_SIZE = 128
steps = EPOCHS * int(np.ceil(0.9 * len(x_train) / BATCH_SIZE))
optimizer = keras.optimizers.AdamW(learning_rate=keras.optimizers.schedules.CosineDecay(1e-3, steps),
                                   weight_decay=0.05)
strong_cnn.compile(optimizer=optimizer, loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
                   metrics=["accuracy"])
history_strong = strong_cnn.fit(x_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.1)

test_loss, test_acc = strong_cnn.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.3f}")

if 'history_shallow' in globals():  # the small CNN was trained in this session
  plt.plot(history_shallow.history['val_accuracy'], label='small CNN')
plt.plot(history_strong.history['val_accuracy'], label='strong CNN')
plt.title('Validation accuracy')
plt.xlabel('epoch')
plt.legend()
plt.show()

save_model("strong_cnn", strong_cnn, history_strong, len(x_train), [32, 32],
           "RGB image resized to 32x32, pixel values divided by 255.")

# Understanding deep networks

*   What is the use of activation functions in network? Why is it needed?
*   We have used softmax activation function in the exercise. There are other activation functions available too. What is the difference between sigmoid activation and softmax activation?
*   What is the difference between categorical crossentropy and binary crossentropy loss?

# Transfer Learning

It is not always necessary to train models from scratch. We can use the knowledge of networks trained on other tasks to learn the task at hand. In this exercise, we will explore the use of pre-trained weights and train on the CIFAR-10 dataset.

*   Create a base imagenet pretrained InceptionV3 model.
    *    Hint: Use tf.keras.applications to create the model
    *    Pay attention to the include_top parameter.
*    Add a global average pooling layer, followed by a fully-connected layer with 1024 neurons and then the classification layer.
*   Train the model by freezing the base model. Train only the newly added layers.
    *    Hint: Every layer has an attribute called 'trainable'
*   Compile the model and train the model for a few epochs only.

In [ ]:
from keras.backend import clear_session
clear_session()

In [ ]:
# solution
from keras.applications import InceptionV3
from tensorflow.keras.optimizers import Adam
from keras.layers import Dense, GlobalAveragePooling2D, Input, RandomFlip, Rescaling, Resizing
from keras.models import Model

INCEPTION_SIZE = 128
inception_model = InceptionV3(
    include_top=False,
    weights="imagenet",
    input_shape=(INCEPTION_SIZE, INCEPTION_SIZE, 3),
    name="inception_v3",
)

# freeze the base model
for layer in inception_model.layers:
    layer.trainable = False

# The model takes the 32x32 images in [0, 1] (as the app gives them), upsamples them and rescales them
# to [-1, 1], the range InceptionV3's ImageNet weights were trained with.
inputs = Input(shape=(32, 32, 3))
x = RandomFlip("horizontal")(inputs)  # only active while training
x = Resizing(INCEPTION_SIZE, INCEPTION_SIZE)(x)
x = Rescaling(2.0, offset=-1.0)(x)
x = inception_model(x, training=False)  # keep the frozen batch norm statistics
x = GlobalAveragePooling2D(name="global_avg_pool")(x)
x = Dense(1024, activation='relu', name="custom_dense")(x)
predictions = Dense(10, activation='softmax', name="predictions")(x)

model = Model(inputs=inputs, outputs=predictions)

learning_rate = 0.001
model.compile(optimizer=Adam(learning_rate=learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
# model.summary()

In [ ]:
# solution: train the new layers on the whole training set (the model resizes the images itself)
history_inception = model.fit(x_train, y_train, epochs=8, batch_size=128, validation_split=0.1)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.3f}")

In [ ]:
# Set 8.1: save the trained weights and the model passport
save_model("inception_v3", model, history_inception, len(x_train), [32, 32],
           "RGB image resized to 32x32, pixel values divided by 255. The model itself upsamples to "
           f"{INCEPTION_SIZE}x{INCEPTION_SIZE} and rescales to [-1, 1], as InceptionV3 expects.")

*    Freeze the bottom layers and unfreeze the base layers.
*    Compile and train the classifier with a very low learning rate (0.0001)

In [ ]:
for layer in inception_model.layers:
  layer.trainable = True
for layer in model.layers:
  if layer.name in ['custom_dense', 'predictions']:
    layer.trainable = False
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

*    When do we train models from scratch? What are the potential issues in training models from scratch?

*    Why do we use pre-trained weights?
*    What is the difference between using random initialization and using weights from a pre-trained model?

# Extracting features from Deep Networks

It is quite possible to extract features (similar to SIFT or ORB) from different layers of deep network.

*   Load ResNet50 model with imagenet weights and check the summary of the model
*   Create a model to extract features from the 'avg_pool' layer.
*   Extract features from the layer for all the train images.
*   Use the extracted features to train a SVM classifier.
    *    Use GridSearchCV and SVC to perform the classification

In [ ]:
# solution
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import Model

model = ResNet50()
model.summary()

*    Evaluate the trained SVM model using the test set
*    Calculate the accuracy score and confusion matrix

In [ ]:
# solution
from skimage import transform
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess

extract = Model(model.inputs, model.get_layer('avg_pool').output)

dl_feature_descriptors = []

N = 50
# get N examples per class
x_train_stratified = np.empty((10*N,)+x_train[0].shape)
y_train_stratified = np.empty((10*N,)+y_train[0].shape)
for cl in range(10):
  x_train_stratified[cl*N:(cl+1)*N] = x_train[np.where(np.argmax(y_train, axis=1) == cl)[0][:N]]
  y_train_stratified[cl*N:(cl+1)*N] = y_train[np.where(np.argmax(y_train, axis=1) == cl)[0][:N]]
print(np.unique(np.argmax(y_train_stratified, axis=1), return_counts=True))

# resize to 224x224x3
x_train_stratified = np.array([
  transform.resize(img, (224, 224, 3), preserve_range=True, anti_aliasing=True) 
  for img in x_train_stratified
  ])
batch_size = 32
for i in range(0, len(x_train_stratified), batch_size):
  example = x_train_stratified[i:i+batch_size]
  # ResNet50 expects 0-255 BGR pixels minus the ImageNet mean, not values in [0, 1]
  desc = extract.predict(resnet50_preprocess(example * 255), verbose=0)
  dl_feature_descriptors.append(desc)

dl_feature_descriptors = np.concatenate(dl_feature_descriptors, axis=0)
print(dl_feature_descriptors.shape)

In [ ]:
# Set 8.3: extract the features of the whole test set (10000 images, 1000 per class) with each model and save them
test_labels = np.argmax(y_test, axis=1)
test_ids = np.arange(len(x_test))

def resize(images, size):
  return np.array([transform.resize(img, (size, size, 3), preserve_range=True, anti_aliasing=True) for img in images])

def save_features(name, extractor, size=32, preprocess=None):
  # resize and predict in batches, so the 224x224 copies are never all in memory at once
  features = []
  for start in range(0, len(test_ids), 250):
    images = x_test[test_ids[start:start + 250]]
    if size != 32:
      images = resize(images, size)
    if preprocess is not None:
      images = preprocess(images)
    features.append(extractor.predict(images, batch_size=batch_size, verbose=0))
  features = np.concatenate(features)
  np.savez(f"{OUTPUT}/{name}.test_features.npz",
           features=features, labels=test_labels[test_ids], class_names=CLASS_NAMES)
  print(name, features.shape)

# our trained models: the dense layer just before the softmax, at the input size their passport gives
for name in ["strong_cnn", "inception_v3"]:
  trained = keras.models.load_model(f"{OUTPUT}/{name}.keras")
  with open(f"{OUTPUT}/{name}.passport.json") as f:
    size = json.load(f)["input_size"][0]
  save_features(name, Model(trained.inputs, trained.layers[-2].output), size=size)

# ImageNet ResNet50: its avg_pool layer (extract, defined above), with ResNet50's own preprocessing
save_features("resnet50", extract, size=224, preprocess=lambda images: resnet50_preprocess(images * 255))

# Feature Visualizations

In order to visualize the features of a higher dimension data, t-SNE is used. t-SNE converts the affinities of the data points to probabilities. It recreates the probability distribution in a low-dimensional space. It is very helpful in visualizing features of different layers in a neural network.

You can find more information about t-SNE [here](https://scikit-learn.org/stable/modules/manifold.html#t-distributed-stochastic-neighbor-embedding-t-sne)

*    Use TSNE to visualize the features extracted in the previous exercise.
    *    Hint: TSNE function is available in the *sklearn.manifold* package.

In [ ]:
# solution
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, verbose=1, perplexity=40, max_iter=300)
tsne_results = tsne.fit_transform(dl_feature_descriptors)

plt.figure(figsize=(10, 10))
for cl in range(10):
  plt.scatter(tsne_results[np.where(np.argmax(y_train_stratified, axis=1) == cl)][:, 0],
              tsne_results[np.where(np.argmax(y_train_stratified, axis=1) == cl)][:, 1],
              label=cl)
plt.show()


*    Why is feature visualization helpful?

# Set 8: download the results

Extract the zip, then use the `.keras` files in **CNN classification** and the `.test_features.npz` file in **Feature projection** in the ForensicsApp.

In [ ]:
import shutil
from google.colab import files

files.download(shutil.make_archive(OUTPUT, "zip", OUTPUT))